# WAI-illustrious SDXL — tạo ảnh trên Google Colab

Chạy **checkpoint SDXL đầy đủ** (`.safetensors`) bằng Diffusers, không cần WebUI hoặc đường link Gradio công khai. Notebook không chứa hay tự tải trọng số model.

**Bắt đầu:**
1. Vào **Runtime → Change runtime type → GPU** (T4 hoặc GPU mạnh hơn).
2. Tải checkpoint từ [trang WAI-illustrious-SDXL](https://civitai.com/models/827184/wai-illustrious-sdxl) theo điều khoản của tác giả. Lưu vào Google Drive, ví dụ `MyDrive/AI/models/WAI-illustrious.safetensors`. **Không cần đổi tên file:** nếu tên tải về khác, sửa `MODEL_PATH` ở ô 3 cho đúng. Dùng checkpoint đầy đủ, không phải LoRA/UNet-only/FP8.
3. Chạy lần lượt các ô **1 → 5**, cho phép Colab kết nối Google Drive, sau đó sửa prompt và chạy lại **ô 5** để tạo ảnh khác.

Mặc định ảnh PNG được lưu trong `MyDrive/AI/outputs` (đổi được ở ô 3). GPU, dung lượng và thời gian chạy tùy hạn mức tài khoản Colab. Lần đầu có thể cần Internet để lấy cấu hình/tokenizer SDXL từ Hugging Face; không tải thêm trọng số SDXL base.

In [ ]:
# @title 1. Kiểm tra GPU
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Chưa có GPU. Chọn Runtime → Change runtime type → GPU, rồi chạy lại ô này.")
device = torch.cuda.get_device_properties(0)
free_bytes, total_bytes = torch.cuda.mem_get_info()
print(f"GPU: {device.name} | VRAM trống: {free_bytes / 2**30:.1f}/{total_bytes / 2**30:.1f} GiB")
print(f"PyTorch: {torch.__version__}")

In [ ]:
# @title 2. Cài thư viện (giữ nguyên PyTorch/CUDA có sẵn của Colab)
%pip -q install "diffusers==0.35.2" "transformers==4.52.4" "accelerate==1.10.1" "safetensors>=0.4.5,<1" "huggingface-hub>=0.34,<1"

In [ ]:
# @title 3. File model, Drive và bộ nhớ { display-mode: "form" }
MOUNT_DRIVE = True # @param {type:"boolean"}
MODEL_PATH = "/content/drive/MyDrive/AI/models/WAI-illustrious.safetensors" # @param {type:"string"}
CACHE_MODEL_LOCAL = True # @param {type:"boolean"}
OUTPUT_DIR = "/content/drive/MyDrive/AI/outputs" # @param {type:"string"}
VRAM_MODE = "auto" # @param ["auto", "speed", "low_vram"]

import os
import shutil
from pathlib import Path
from safetensors import safe_open

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

source_model = Path(MODEL_PATH).expanduser()
output_dir = Path(OUTPUT_DIR).expanduser()
if not source_model.is_absolute() or not output_dir.is_absolute():
    raise ValueError("MODEL_PATH và OUTPUT_DIR phải là đường dẫn tuyệt đối (bắt đầu bằng /content/...).")
drive_root = Path("/content/drive")
using_drive = any(drive_root == p or drive_root in p.parents for p in (source_model, output_dir))
if using_drive and not Path("/content/drive/MyDrive").is_dir():
    raise RuntimeError("Google Drive chưa được gắn. Bật MOUNT_DRIVE rồi chạy lại ô 3.")
if not source_model.is_file():
    raise FileNotFoundError(f"Không tìm thấy model: {source_model}. Kiểm tra tên file/đường dẫn trong Drive.")
if source_model.suffix.lower() != ".safetensors":
    raise ValueError("MODEL_PATH phải trỏ tới checkpoint .safetensors, không phải thư mục hoặc LoRA.")
model_size = source_model.stat().st_size
if model_size < 100 * 2**20:
    raise ValueError("File quá nhỏ để là checkpoint SDXL đầy đủ; có thể tải thiếu hoặc là trang HTML/LoRA.")

# Chỉ đọc header safetensors; không nạp toàn bộ trọng số vào RAM.
try:
    with safe_open(str(source_model), framework="pt", device="cpu") as model_header:
        keys = model_header.keys()
        has_unet = any(key.startswith("model.diffusion_model.") for key in keys)
        has_text_encoders = any(key.startswith("conditioner.embedders.") for key in keys)
except Exception as exc:
    raise ValueError("Không mở được file safetensors. Kiểm tra file đã tải đủ và đúng định dạng.") from exc
if not (has_unet and has_text_encoders):
    raise ValueError("Cần checkpoint SDXL đầy đủ (UNet và text encoder), không phải LoRA/UNet-only.")

checkpoint = source_model
# Nếu đã upload vào /content (không phải Drive) thì không sao chép lần nữa.
already_local = Path("/content") in source_model.parents and drive_root not in source_model.parents
if CACHE_MODEL_LOCAL and not already_local:
    cache_dir = Path("/content/wai_model_cache")
    cache_dir.mkdir(parents=True, exist_ok=True)
    cached_model = cache_dir / source_model.name
    if cached_model != source_model:
        if not cached_model.is_file() or cached_model.stat().st_size != model_size:
            if shutil.disk_usage(cache_dir).free < model_size + 2 * 2**30:
                raise OSError("Không đủ bộ nhớ đĩa để sao chép model. Tắt CACHE_MODEL_LOCAL ở ô 3.")
            partial = cached_model.with_name(cached_model.name + ".partial")
            try:
                print("Đang sao chép checkpoint sang ổ /content để nạp nhanh hơn...")
                shutil.copyfile(source_model, partial)
                if partial.stat().st_size != model_size:
                    raise OSError("Bản sao model không đầy đủ; hãy thử lại.")
                os.replace(partial, cached_model)
            finally:
                partial.unlink(missing_ok=True)
        checkpoint = cached_model
if VRAM_MODE not in ("auto", "speed", "low_vram"):
    raise ValueError("VRAM_MODE phải là auto, speed hoặc low_vram.")
print(f"Checkpoint: {checkpoint} ({model_size / 2**30:.2f} GiB)")
print(f"Ảnh sẽ lưu tại: {output_dir}")

In [ ]:
# @title 4. Nạp model (chạy một lần; sau đó chỉ chạy lại ô 5)
import gc
from diffusers import EulerAncestralDiscreteScheduler, StableDiffusionXLPipeline

# Giải phóng pipeline cũ nếu người dùng đổi VRAM_MODE và chạy lại ô này.
if "pipe" in globals():
    del pipe
    gc.collect()
    torch.cuda.empty_cache()

torch.backends.cuda.matmul.allow_tf32 = True  # có tác dụng trên GPU hỗ trợ TF32
torch.backends.cudnn.allow_tf32 = True
free_bytes, _ = torch.cuda.mem_get_info()
use_offload = VRAM_MODE == "low_vram" or (VRAM_MODE == "auto" and free_bytes < 14 * 2**30)
print("Chế độ:", "tiết kiệm VRAM (CPU offload)" if use_offload else "ưu tiên tốc độ (FP16 trên GPU)")

pipe = StableDiffusionXLPipeline.from_single_file(
    str(checkpoint),
    torch_dtype=torch.float16,
    use_safetensors=True,
)
pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe.scheduler.config)  # Euler a
pipe.vae.enable_slicing()
if use_offload:
    pipe.vae.enable_tiling()
    pipe.enable_model_cpu_offload()
else:
    pipe.to("cuda")
print("Model đã sẵn sàng. Hãy chạy ô 5 để tạo ảnh.")

In [ ]:
# @title 5. Tạo ảnh (sửa prompt/seed và chạy lại tùy thích) { display-mode: "form" }
PROMPT = "general, 1girl, solo, cherry blossoms, spring, soft sunlight, detailed eyes, anime illustration, masterpiece, best quality" # @param {type:"string"}
NEGATIVE_PROMPT = "nsfw, explicit, lowres, worst quality, bad anatomy, blurry" # @param {type:"string"}
SIZE = "1024x1024" # @param ["1024x1024", "832x1216", "1216x832", "768x1024", "1024x768"]
STEPS = 25 # @param {type:"slider", min:10, max:45, step:1}
CFG = 6.0 # @param {type:"slider", min:1, max:12, step:0.5}
SEED = -1 # @param {type:"integer"}
EMBED_METADATA = True # @param {type:"boolean"}

import json
import secrets
from datetime import datetime, timezone
from IPython.display import display
from PIL.PngImagePlugin import PngInfo

if "pipe" not in globals():
    raise RuntimeError("Chưa nạp model. Hãy chạy ô 4 trước.")
if not PROMPT.strip():
    raise ValueError("PROMPT không được để trống.")
width, height = map(int, SIZE.split("x"))
if min(width, height) < 512 or width % 8 or height % 8:
    raise ValueError("Kích thước ảnh phải >= 512 và chia hết cho 8.")
if not 1 <= STEPS <= 50 or not 1 <= CFG <= 12:
    raise ValueError("STEPS phải từ 1–50 và CFG từ 1–12.")
if SEED != -1 and not 0 <= SEED < 2**32:
    raise ValueError("SEED phải là -1 (ngẫu nhiên) hoặc số nguyên từ 0 đến 2^32 - 1.")
seed = secrets.randbelow(2**32) if SEED == -1 else SEED
generator = torch.Generator(device="cpu").manual_seed(seed)

try:
    with torch.inference_mode():
        image = pipe(
            prompt=PROMPT.strip(),
            negative_prompt=NEGATIVE_PROMPT.strip(),
            width=width,
            height=height,
            num_inference_steps=STEPS,
            guidance_scale=CFG,
            num_images_per_prompt=1,
            generator=generator,
        ).images[0]
except torch.cuda.OutOfMemoryError as exc:
    torch.cuda.empty_cache()
    raise RuntimeError(
        "Hết VRAM: khởi động lại runtime, chọn VRAM_MODE='low_vram' ở ô 3 "
        "hoặc giảm kích thước ảnh, rồi chạy lại các ô."
    ) from exc

display(image)
if (output_dir == drive_root or drive_root in output_dir.parents) and not Path("/content/drive/MyDrive").is_dir():
    raise RuntimeError("Drive đã ngắt kết nối; kết nối lại ở ô 3 trước khi lưu ảnh.")
output_dir.mkdir(parents=True, exist_ok=True)
filename = f"wai_{datetime.now(timezone.utc):%Y%m%d_%H%M%S_%f}_{seed}.png"
output_path = output_dir / filename
png_info = PngInfo()
if EMBED_METADATA:
    png_info.add_text("parameters", json.dumps({
        "model": source_model.name,
        "prompt": PROMPT.strip(),
        "negative_prompt": NEGATIVE_PROMPT.strip(),
        "seed": seed,
        "width": width,
        "height": height,
        "steps": STEPS,
        "cfg": CFG,
        "sampler": "Euler a",
    }, ensure_ascii=False))
image.save(output_path, pnginfo=png_info)
print(f"Seed: {seed} | Đã lưu: {output_path}")

### Mẹo và xử lý sự cố

- **Không tìm thấy model:** kiểm tra chính xác `MODEL_PATH` (kể cả đuôi file) và quyền gắn Drive. Nếu upload tạm vào Colab, đặt `MODEL_PATH` ở `/content/...` và `CACHE_MODEL_LOCAL=False`.
- **Hết VRAM:** khởi động lại runtime, chọn `VRAM_MODE=low_vram`; nếu vẫn thiếu, dùng `768x1024`/`1024x768`. `auto` dùng GPU trực tiếp khi VRAM trống ≥ 14 GiB, ngược lại offload qua CPU (chậm hơn).
- **Thiếu dung lượng đĩa:** tắt `CACHE_MODEL_LOCAL` để nạp trực tiếp từ Drive. Bản sao trong `/content` chỉ tồn tại trong phiên Colab hiện tại.
- **Lỗi khi tải checkpoint:** cần file SDXL `.safetensors` đầy đủ, không phải LoRA hoặc bản FP8 chỉ dành cho công cụ khác. Lần đầu `from_single_file` cần truy cập mạng để lấy cấu hình/tokenizer.
- `SEED=-1` chọn seed ngẫu nhiên; nhập một seed cụ thể để thử lặp lại với cùng prompt/cấu hình. Chỉ sinh **một ảnh mỗi lượt** để tiết kiệm VRAM. PNG có thể chứa prompt trong metadata khi `EMBED_METADATA=True`; cân nhắc tắt trước khi chia sẻ.
- Prompt/negative mặc định hướng tới nội dung lành mạnh nhưng **không đảm bảo lọc nội dung**; tự kiểm tra ảnh và tuân thủ điều khoản của model và Colab. Không chia sẻ file Drive hoặc notebook đã sửa nếu chứa dữ liệu riêng tư.